# 🎯 DGNet: Deep Gradient Network Model, S3 Ingestion, Training & MLflow Tracking
### Dual-Branch Architecture, Remote S3 Ingestion, Dedicated MLflow Tracker, COD Benchmark & Edge Export

This notebook provides the complete **DGNet (Deep Gradient Network)** model execution pipeline for **remote execution environments** (e.g. Google Colab, Kaggle, AWS EC2/SageMaker).
It matches `scripts/DGnet.py` and `scripts/hf_sync.py` exactly.

#### Pipeline Modules:
1. **Environment Setup & Hardware Detection**: GPU memory growth and OpenCV thread safety.
2. **S3 / Cloud Remote Dataset Ingestion**: Automatic dataset download and extraction directly inside the notebook.
3. **Configuration & Hyperparameters**: Centralized project parameters (`params.yaml`).
4. **Robust Dataset Ingestion & Merging**: Discovers raw splits and merges offline tactical augmented samples.
5. **High-Throughput Multi-Scale `tf.data` Pipeline**: Geometric transforms, tactical augmentations (smoke, thermal, defocus, cutout), and ImageNet standardization.
6. **DGNet Architecture**: Multi-scale Context Encoder + Canny Texture Encoder + GIT Cross-Attention + NCD Decoder with Deep Supervision.
7. **Dedicated Notebook MLflow Tracker**: Fresh, self-contained tracker supporting local SQLite (`metrics/mlflow_db/mlflow.db`) and remote S3 artifact storage.
8. **Loss Function & Optimization Graph**: Composite Weighted BCE + Weighted IoU + Canny Edge BCE (`DGNetLoss`), AdamW optimizer, and WarmupCosineDecay schedule.
9. **Interactive Training Loop with Live MLflow Tracking**: Batch-level loss tracking every 10 batches, epoch-level validation, and checkpoint artifact logging.
10. **Official COD Benchmark Evaluation**: Evaluates test split and logs official metrics (Pixel Acc, MAE, IoU, Dice, $F_\beta$, Precision, Recall, Specificity, FPS) directly to MLflow.
11. **Clean Black-and-White Visual Comparison**: Pure binary masks (0 and 255, no color heatmaps) labeled with model names, saved and logged as MLflow artifacts.
12. **TFLite INT8 Edge Quantization**: Quantization to `models/quantized/dgnet_s_int8.tflite` with MLflow artifact registration.
13. **S3 / Remote Checkpoint Push**: Synchronize best checkpoints and MLflow logs back to S3 / HF Storage Bucket.
14. **MLflow UI & Run Query**: Queries database and displays logged runs with UI launch instructions.


In [1]:
# 1. Environment Setup & Hardware Configuration
%load_ext autoreload
%autoreload 2

import os
import sys
import time
import shutil
from pathlib import Path

import cv2
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import mlflow
from mlflow.tracking import MlflowClient

# Load environment variables from .env if present
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT.resolve()}")
print(f"TensorFlow:   {tf.__version__}")
print(f"MLflow:       {mlflow.__version__}")

# Automatic GPU Hardware Detection & Memory Growth
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
            print(f"✓ GPU Detected: {gpu.name} (Memory growth enabled)")
        except Exception as e:
            print(f"Notice (Memory Growth): {e}")
else:
    print("ℹ No physical GPU detected; running on CPU.")

from src.models.dgnet import DGNet, DGNetLoss, build_dgnet_model
from src.utils.config import load_project_config
from src.utils.metrics import CODMetricsAccumulator, compute_all_metrics
from src.data.augmentation import tf_random_military_fog_smoke, tf_random_thermal_noise


c:\Users\Acer\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project Root: C:\Users\Acer\Desktop\COD_PROJECT
TensorFlow:   2.19.0
MLflow:       3.10.0
ℹ No physical GPU detected; running on CPU.


## 2. S3 / Cloud Remote Dataset Ingestion
Since execution is performed outside the local system (e.g. on Google Colab, Kaggle, AWS EC2, or remote GPU server), this cell automatically downloads and extracts the raw dataset from **S3 / Hugging Face Storage Bucket** into `data/raw/dataset-splitM`.
- Compatible with Hugging Face S3 Gateway (`https://s3.hf.co/gouravbirwaz` / `gouravbirwaz/COD_dataset`).
- Compatible with standard AWS S3 buckets (`boto3`).
- Skips download automatically if the dataset is already present.


In [ ]:
# S3 & Remote Storage Configuration
S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME", "COD_dataset")
S3_ENDPOINT_URL = os.getenv("AWS_ENDPOINT_URL_S3", os.getenv("MLFLOW_S3_ENDPOINT_URL", "https://s3.hf.co/gouravbirwaz"))
HF_REPO_ID = os.getenv("HF_REPO_ID", "gouravbirwaz/COD_dataset")
HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN", None)

def fetch_dataset_from_s3(target_dir: Path, repo_id: str = HF_REPO_ID, hf_token: str = HF_TOKEN, force: bool = False):
    """Fetches raw COD dataset from S3 / Hugging Face storage bucket if not already present."""
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    
    existing_imgs = list(target_dir.rglob("*.jpg")) + list(target_dir.rglob("*.png")) + list(target_dir.rglob("*.jpeg"))
    if existing_imgs and not force:
        print(f"✓ Dataset already present at: {target_dir.resolve()} ({len(existing_imgs)} images found). Skipping download.")
        return True
        
    print(f"\n[S3 Fetch] Dataset missing in {target_dir.resolve()}. Initiating cloud remote download...")
    
    # 1. Hugging Face S3 Gateway / Hub Download
    try:
        from huggingface_hub import hf_hub_download, snapshot_download
        print(f"  • Attempting archive download from '{repo_id}' (data/dataset_raw.zip)...")
        try:
            zip_file = hf_hub_download(
                repo_id=repo_id,
                repo_type="dataset",
                filename="data/dataset_raw.zip",
                local_dir=str(PROJECT_ROOT),
                token=hf_token
            )
            if zip_file and Path(zip_file).exists():
                print(f"  • Unpacking dataset archive {zip_file}...")
                import zipfile
                extract_dest = target_dir.parent if target_dir.name == "dataset-splitM" else target_dir
                with zipfile.ZipFile(zip_file, 'r') as zf:
                    zf.extractall(extract_dest)
                print(f"✓ SUCCESS: Extracted raw dataset to {extract_dest.resolve()}!")
                return True
        except Exception as e_zip:
            print(f"  ! Archive download notice: {e_zip}. Trying snapshot sync...")
            
        snapshot_download(
            repo_id=repo_id,
            repo_type="dataset",
            local_dir=str(PROJECT_ROOT),
            allow_patterns="data/raw/*",
            token=hf_token
        )
        print(f"✓ SUCCESS: Snapshot synced to {target_dir.resolve()}!")
        return True
    except Exception as e_hf:
        print(f"  ! Hugging Face fetch notice: {e_hf}")
        
    # 2. Boto3 S3 Client Download
    try:
        import boto3
        from botocore import UNSIGNED
        from botocore.config import Config
        print(f"  • Attempting Boto3 S3 fetch from endpoint: {S3_ENDPOINT_URL}...")
        s3_kwargs = {"endpoint_url": S3_ENDPOINT_URL} if S3_ENDPOINT_URL else {}
        if os.getenv("AWS_ACCESS_KEY_ID") and os.getenv("AWS_SECRET_ACCESS_KEY"):
            s3_kwargs["aws_access_key_id"] = os.getenv("AWS_ACCESS_KEY_ID")
            s3_kwargs["aws_secret_access_key"] = os.getenv("AWS_SECRET_ACCESS_KEY")
        else:
            s3_kwargs["config"] = Config(signature_version=UNSIGNED)
        
        s3 = boto3.client("s3", **s3_kwargs)
        dl_zip = PROJECT_ROOT / "data" / "dataset_raw.zip"
        s3.download_file(S3_BUCKET_NAME, "data/dataset_raw.zip", str(dl_zip))
        if dl_zip.exists() and dl_zip.stat().st_size > 0:
            import zipfile
            extract_dest = target_dir.parent if target_dir.name == "dataset-splitM" else target_dir
            with zipfile.ZipFile(dl_zip, 'r') as zf:
                zf.extractall(extract_dest)
            print(f"✓ SUCCESS: Downloaded and extracted via Boto3 S3!")
            return True
    except Exception as e_s3:
        print(f"  ! Boto3 S3 fetch notice: {e_s3}")
        
    return False

DATA_DIR = PROJECT_ROOT / "data/raw/dataset-splitM"
AUGMENTED_DIR = PROJECT_ROOT / "data/augmented/military_tactical"
fetch_dataset_from_s3(DATA_DIR)


## 3. Configuration & Hyperparameters
Loads parameters from `params.yaml` and sets up target directories.


In [ ]:
cfg = load_project_config()
data_cfg = cfg.get("data", cfg.get("dataset", {}))
model_cfg = cfg.get("model", {})
train_cfg = cfg.get("train", {})

IMG_SIZE = tuple(data_cfg.get("image_size", [384, 384]))
BATCH_SIZE = train_cfg.get("batch_size", 8)
LEARNING_RATE = train_cfg.get("learning_rate", 1e-4)
EPOCHS = train_cfg.get("epochs", 50)
BACKBONE = model_cfg.get("context_backbone", "mobilenet_v3_large")
GRADIENT_KERNEL = model_cfg.get("gradient_kernel", "sobel")
MODEL_VERSION = model_cfg.get("version", "v2.0_official_dgnet")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "COD-Camouflage-Detection")

CHECKPOINT_DIR = PROJECT_ROOT / "models/raw_checkpoints"
METRICS_DIR = PROJECT_ROOT / "metrics"
PRED_DIR = PROJECT_ROOT / "experiments/predictions"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment Name: {EXPERIMENT_NAME}")
print(f"Model Version:   {MODEL_VERSION}")
print(f"Backbone:        {BACKBONE}")
print(f"Gradient Kernel: {GRADIENT_KERNEL}")
print(f"Image Size:      {IMG_SIZE}")
print(f"Batch Size:      {BATCH_SIZE}")
print(f"Learning Rate:   {LEARNING_RATE}")
print(f"Epochs:          {EPOCHS}")


## 4. Robust Dataset Pipeline Ingestion
Matching `scripts/DGnet.py`: discovers raw dataset splits (`Training`, `Testing`) and automatically merges offline tactical augmented samples (`data/augmented/military_tactical`).


In [ ]:
def find_image_mask_pairs(base_dir: Path, split_name: str):
    """Finds matched image and mask pairs supporting case-insensitive paths."""
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    split_dir = base_dir / split_name
    if not split_dir.exists():
        return [], []
        
    img_candidates = [split_dir / "images", split_dir / "Images", split_dir / "rgb", split_dir]
    gt_candidates = [split_dir / "GT", split_dir / "gt", split_dir / "Masks", split_dir / "masks"]
    
    img_dir = next((d for d in img_candidates if d.exists() and d.is_dir()), None)
    gt_dir = next((d for d in gt_candidates if d.exists() and d.is_dir()), None)
    
    if not img_dir or not gt_dir:
        return [], []
        
    imgs = sorted([str(p) for p in img_dir.iterdir() if p.suffix.lower() in valid_exts])
    matched_imgs, matched_masks = [], []
    for ip in imgs:
        stem = Path(ip).stem
        for ext in [".png", ".jpg", ".jpeg", ".bmp"]:
            gp = gt_dir / f"{stem}{ext}"
            if gp.exists():
                matched_imgs.append(ip)
                matched_masks.append(str(gp))
                break
    return matched_imgs, matched_masks

# Discover raw training and testing sets
raw_train_imgs, raw_train_masks = find_image_mask_pairs(DATA_DIR, "Training")
test_imgs, test_masks = find_image_mask_pairs(DATA_DIR, "Testing")

# Discover and merge augmented training samples (from DVC/S3 or generator)
aug_train_imgs, aug_train_masks = find_image_mask_pairs(AUGMENTED_DIR, "Training")

train_imgs = raw_train_imgs + aug_train_imgs
train_masks = raw_train_masks + aug_train_masks

print(f"✓ Loaded Raw Training Samples:       {len(raw_train_imgs)}")
print(f"✓ Loaded Augmented Training Samples: {len(aug_train_imgs)}")
print(f"✓ Total Combined Training Samples:   {len(train_imgs)}")
print(f"✓ Loaded Testing Samples:            {len(test_imgs)}")


## 5. High-Throughput Multi-Scale & Tactical `tf.data` Pipeline
Matching `scripts/DGnet.py`: applies official Canny object-level gradient extraction, multi-scale random resized crops, geometric flips/rotations, tactical military environmental augmentations (fog, thermal grain, blur), random erasing/cutout, and ImageNet standardization.


In [ ]:
IMAGENET_MEAN = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
IMAGENET_STD = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)

def compute_object_gradient_np(image_uint8: np.ndarray, mask_uint8: np.ndarray) -> np.ndarray:
    """Official DGNet Canny object gradient supervision (cv2.Canny * mask)."""
    if image_uint8.ndim == 4:
        image_uint8 = image_uint8[0]
    if mask_uint8.ndim == 3 and mask_uint8.shape[-1] == 1:
        mask_uint8 = mask_uint8[:, :, 0]
    canny = cv2.Canny(image_uint8, 64, 128)
    fore = ((canny > 0) & (mask_uint8 >= 128)).astype(np.float32)
    return fore[:, :, np.newaxis]

def parse_sample(img_path: tf.Tensor, mask_path: tf.Tensor, image_size: tuple):
    img_bytes = tf.io.read_file(img_path)
    image_raw = tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
    image = tf.image.resize(image_raw, image_size)
    image_uint8 = tf.cast(image, tf.uint8)

    mask_bytes = tf.io.read_file(mask_path)
    mask = tf.image.decode_image(mask_bytes, channels=1, expand_animations=False)
    mask = tf.image.resize(mask, image_size, method="nearest")
    mask_bin = tf.cast(mask >= 128, tf.float32)

    def _py_grad(im, m):
        return compute_object_gradient_np(im, m)

    grad_gt = tf.numpy_function(_py_grad, [image_uint8, tf.cast(mask, tf.uint8)], tf.float32)
    grad_gt.set_shape([image_size[0], image_size[1], 1])

    image_float = tf.cast(image, tf.float32) / 255.0
    image_float.set_shape([image_size[0], image_size[1], 3])
    mask_bin.set_shape([image_size[0], image_size[1], 1])

    return image_float, mask_bin, grad_gt

def _random_resized_crop(img, mask, grad_gt, target_size, scale_min=0.5, scale_max=1.0):
    """Multi-scale Random Resized Crop applied identically across image, mask, and gradient GT."""
    h, w = target_size
    img_h, img_w = tf.shape(img)[0], tf.shape(img)[1]
    scale = tf.random.uniform((), minval=scale_min, maxval=scale_max)
    crop_h = tf.minimum(tf.cast(tf.cast(img_h, tf.float32) * tf.sqrt(scale), tf.int32), img_h)
    crop_w = tf.minimum(tf.cast(tf.cast(img_w, tf.float32) * tf.sqrt(scale), tf.int32), img_w)
    offset_h = tf.random.uniform((), minval=0, maxval=img_h - crop_h + 1, dtype=tf.int32)
    offset_w = tf.random.uniform((), minval=0, maxval=img_w - crop_w + 1, dtype=tf.int32)
    img = tf.image.crop_to_bounding_box(img, offset_h, offset_w, crop_h, crop_w)
    mask = tf.image.crop_to_bounding_box(mask, offset_h, offset_w, crop_h, crop_w)
    grad_gt = tf.image.crop_to_bounding_box(grad_gt, offset_h, offset_w, crop_h, crop_w)
    return tf.image.resize(img, [h, w]), tf.image.resize(mask, [h, w], method="nearest"), tf.image.resize(grad_gt, [h, w], method="nearest")

def _cutout_tf(img, max_fraction=0.25):
    """Random rectangular patch erasing (CutOut) to reduce overfitting."""
    h, w = tf.shape(img)[0], tf.shape(img)[1]
    cut_h = tf.cast(tf.cast(h, tf.float32) * tf.random.uniform((), 0.05, max_fraction), tf.int32)
    cut_w = tf.cast(tf.cast(w, tf.float32) * tf.random.uniform((), 0.05, max_fraction), tf.int32)
    cy, cx = tf.random.uniform((), minval=0, maxval=h, dtype=tf.int32), tf.random.uniform((), minval=0, maxval=w, dtype=tf.int32)
    y1, y2 = tf.maximum(cy - cut_h // 2, 0), tf.minimum(cy + cut_h // 2, h)
    x1, x2 = tf.maximum(cx - cut_w // 2, 0), tf.minimum(cx + cut_w // 2, w)
    rows, cols = tf.range(h), tf.range(w)
    keep_mask = tf.cast(tf.expand_dims(tf.logical_or(rows < y1, rows >= y2), 1) | tf.expand_dims(tf.logical_or(cols < x1, cols >= x2), 0), tf.float32)
    return img * tf.expand_dims(keep_mask, -1)

def create_tf_dataset(image_paths, mask_paths, image_size=(384, 384), batch_size=8, is_train=True):
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    dataset = dataset.map(lambda x, y: parse_sample(x, y, image_size), num_parallel_calls=tf.data.AUTOTUNE)
    
    if is_train:
        dataset = dataset.shuffle(buffer_size=min(len(image_paths), 256), seed=42)
        
        def _augment_fn(img, mask, grad_gt):
            if tf.random.uniform(()) > 0.5:
                img, mask, grad_gt = _random_resized_crop(img, mask, grad_gt, image_size)
            if tf.random.uniform(()) > 0.5:
                img = tf.image.flip_left_right(img)
                mask = tf.image.flip_left_right(mask)
                grad_gt = tf.image.flip_left_right(grad_gt)
            if tf.random.uniform(()) > 0.5:
                img = tf.image.flip_up_down(img)
                mask = tf.image.flip_up_down(mask)
                grad_gt = tf.image.flip_up_down(grad_gt)
            k = tf.random.uniform((), minval=0, maxval=4, dtype=tf.int32)
            img = tf.image.rot90(img, k=k)
            mask = tf.image.rot90(mask, k=k)
            grad_gt = tf.image.rot90(grad_gt, k=k)
            
            # Military tactical simulations
            if tf.random.uniform(()) > 0.7:
                img = tf_random_military_fog_smoke(img)
            if tf.random.uniform(()) > 0.7:
                img = tf_random_thermal_noise(img)
            if tf.random.uniform(()) > 0.7:
                img = _cutout_tf(img)
                
            img = tf.image.random_brightness(img, max_delta=0.2)
            img = tf.image.random_contrast(img, lower=0.7, upper=1.3)
            img = tf.clip_by_value(img, 0.0, 1.0)
            img_norm = (img - IMAGENET_MEAN) / IMAGENET_STD
            return img_norm, mask, grad_gt
            
        dataset = dataset.map(_augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        dataset = dataset.map(lambda img, m, g: ((img - IMAGENET_MEAN) / IMAGENET_STD, m, g), num_parallel_calls=tf.data.AUTOTUNE)
        
    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

print("✓ tf.data dataset generator function ready.")


## 6. Build DGNet Architecture
Instantiate the dual-branch DGNet model (Context Encoder + Canny Texture Encoder + GIT Cross-Attention + NCD Decoder) and inspect parameter breakdown.


In [ ]:
model = build_dgnet_model(
    backbone_name=BACKBONE,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    gradient_kernel=GRADIENT_KERNEL,
    pretrained=True
)

dummy_input = tf.zeros((1, IMG_SIZE[0], IMG_SIZE[1], 3), dtype=tf.float32)
_ = model(dummy_input, training=False)

total_params = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print("=" * 65)
print(f" DGNet Architecture: {BACKBONE} + {GRADIENT_KERNEL}")
print("=" * 65)
print(f" Total Parameters:      {total_params:,} ({total_params*4 / (1024*1024):.2f} MB)")
print(f" Trainable Parameters:  {trainable_params:,}")
print("=" * 65)


## 7. Dedicated Notebook MLflow Tracker
A fresh, self-contained MLflow experiment tracker connected to `metrics/mlflow_db/mlflow.db`.
Logs hyperparameters, YAML configurations, model architecture summaries, live batch loss metrics, and checkpoints.


In [ ]:
class DGNetNotebookMLflowTracker:
    """
    Self-contained MLflow experiment tracker built specifically for this notebook.
    Persists runs directly to metrics/mlflow_db/mlflow.db or remote tracking server.
    """
    def __init__(self, experiment_name: str = "COD-Camouflage-Detection"): 
        self.experiment_name = experiment_name
        self.active_run = None
        
        # Configure SQLite database path or remote tracking URI
        remote_uri = os.getenv("MLFLOW_TRACKING_URI")
        if remote_uri and not remote_uri.startswith("http://127.0.0.1") and not remote_uri.startswith("http://localhost"):
            self.tracking_uri = remote_uri
        else:
            db_dir = (PROJECT_ROOT / "metrics" / "mlflow_db").resolve()
            db_dir.mkdir(parents=True, exist_ok=True)
            db_path = db_dir / "mlflow.db"
            self.tracking_uri = f"sqlite:///{db_path.as_posix()}"
        
        mlflow.set_tracking_uri(self.tracking_uri)
        mlflow.set_experiment(self.experiment_name)
        print(f"[MLflow] Tracking URI:      {self.tracking_uri}")
        print(f"[MLflow] Active Experiment: '{self.experiment_name}'")
        
    def start_run(self, run_name: str = None, tags: dict = None, description: str = None):
        default_tags = {
            "project": "COD",
            "task": "camouflage_detection",
            "framework": "tensorflow_2.x",
            "notebook": "DGnet.ipynb"
        }
        if tags:
            default_tags.update(tags)
        self.active_run = mlflow.start_run(
            run_name=run_name,
            tags=default_tags,
            description=description
        )
        print(f"[MLflow] ✓ Started Run ID: {self.active_run.info.run_id} (Name: '{run_name}')")
        return self.active_run

    def log_params(self, params: dict):
        if not self.active_run: return
        clean_params = {k: (str(v) if isinstance(v, (list, tuple)) else v) for k, v in params.items()}
        mlflow.log_params(clean_params)

    def log_metrics(self, metrics: dict, step: int = None):
        if not self.active_run: return
        clean_metrics = {k: float(v) for k, v in metrics.items()}
        mlflow.log_metrics(clean_metrics, step=step)

    def log_artifact(self, local_path: str, artifact_path: str = None):
        if not self.active_run: return
        p = Path(local_path)
        if p.exists():
            if p.is_dir():
                mlflow.log_artifacts(str(p), artifact_path=artifact_path)
            else:
                mlflow.log_artifact(str(p), artifact_path=artifact_path)

    def log_configs(self, config_paths: list, artifact_path: str = "configs"):
        for cp in config_paths:
            self.log_artifact(str(cp), artifact_path=artifact_path)

    def log_model_summary(self, model_obj):
        if not self.active_run: return
        summary_path = PROJECT_ROOT / "metrics" / "notebook_model_summary.txt"
        with open(summary_path, "w", encoding="utf-8") as f:
            model_obj.summary(print_fn=lambda x: f.write(x + "\n"))
        self.log_artifact(str(summary_path), artifact_path="architecture")

    def end_run(self, status: str = "FINISHED"):
        if self.active_run:
            run_id = self.active_run.info.run_id
            mlflow.end_run(status=status)
            print(f"[MLflow] ✓ Completed Run ID: {run_id} (Status: {status})")
            self.active_run = None

mlflow_tracker = DGNetNotebookMLflowTracker(experiment_name=EXPERIMENT_NAME)
try:
    mlflow.end_run()
except Exception:
    pass

run_name = f"DGNet-{MODEL_VERSION}-{BACKBONE}"
mlflow_tracker.start_run(
    run_name=run_name,
    tags={
        "model_version": MODEL_VERSION,
        "backbone": BACKBONE,
        "gradient_kernel": GRADIENT_KERNEL,
    },
    description=f"DGNet tactical training and COD evaluation run ({BACKBONE})."
)

mlflow_tracker.log_params({
    "model_name": "DGNet",
    "model_version": MODEL_VERSION,
    "context_backbone": BACKBONE,
    "gradient_kernel": GRADIENT_KERNEL,
    "image_size": f"[{IMG_SIZE[0]}, {IMG_SIZE[1]}]",
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "optimizer": "AdamW",
    "scheduler": "WarmupCosineDecay",
    "loss_function": "DGNetLoss (StructureLoss + MSE)",
    "train_samples": len(train_imgs),
    "test_samples": len(test_imgs),
    "total_parameters": total_params,
    "trainable_parameters": trainable_params
})

config_candidates = [
    PROJECT_ROOT / "params.yaml",
    PROJECT_ROOT / "configs/optimization/train.yaml",
    PROJECT_ROOT / "configs/data/data.yaml",
    PROJECT_ROOT / "configs/model/dgnet.yaml"
]
mlflow_tracker.log_configs([cp for cp in config_candidates if cp.exists()])
mlflow_tracker.log_model_summary(model)


## 8. Loss Function, Optimizer & Deep Supervision Graph
Matching `scripts/DGnet.py`: compiles `DGNetLoss(lambda_grad_loss=1.0, label_smoothing=0.02)` with `WarmupCosineDecay` learning rate schedule and `AdamW` with gradient clipping (`clipnorm=1.0`).


In [ ]:
loss_fn = DGNetLoss(lambda_grad_loss=1.0, label_smoothing=0.02)

# Warmup + Cosine Decay schedule matching scripts/DGnet.py
total_steps = EPOCHS * max(len(train_imgs) // BATCH_SIZE, 1)
warmup_steps = min(10 * max(len(train_imgs) // BATCH_SIZE, 1), total_steps // 5)

class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, initial_lr, warmup_steps, total_steps):
        super().__init__()
        self.initial_lr = initial_lr
        self.warmup_steps = float(warmup_steps)
        self.total_steps = float(total_steps)
        self.cosine_decay = tf.keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=initial_lr,
            decay_steps=int(total_steps - warmup_steps)
        )
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.initial_lr * (step / tf.maximum(self.warmup_steps, 1.0))
        cosine_lr = self.cosine_decay(step - self.warmup_steps)
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

lr_schedule = WarmupCosineDecay(LEARNING_RATE, warmup_steps, total_steps)
optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4, clipnorm=1.0)

@tf.function
def train_step(x_b, y_m, y_g):
    with tf.GradientTape() as tape:
        preds = model(x_b, training=True, return_dict=True)
        preds["gt_grad"] = y_g
        loss = loss_fn(y_m, preds)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

@tf.function
def val_step(x_b, y_m, y_g):
    val_preds = model(x_b, training=False, return_dict=True)
    val_preds["gt_grad"] = y_g
    v_loss = loss_fn(y_m, val_preds)
    prob_mask = tf.nn.sigmoid(val_preds["mask_pred"])
    return v_loss, prob_mask

print("✓ Training and validation step graphs compiled successfully!")


## 9. Model Training Loop with Live MLflow Tracking
Executes training with **live batch-level and epoch-level metric logging in MLflow**:
- `batch_train_loss` & `running_avg_train_loss` logged every 10 batches.
- `epoch_train_loss`, `val_loss`, and `epoch_time_seconds` logged at epoch boundaries.
- Best and latest `.weights.h5` checkpoints logged as MLflow artifacts.


In [ ]:
# Checkpoint Resumption Check
candidate_ckpts = sorted(CHECKPOINT_DIR.glob(f"dgnet_{MODEL_VERSION}*.weights.h5"), reverse=True)
if candidate_ckpts:
    try:
        model.load_weights(str(candidate_ckpts[0]))
        print(f"✓ Loaded existing checkpoint: {candidate_ckpts[0].name}")
    except Exception as e_load:
        print(f"Notice (Weights load): {e_load}")

RUN_TRAINING = False  # Set to True to start training
TRAIN_EPOCHS = 10

if RUN_TRAINING and train_imgs:
    train_dataset = create_tf_dataset(train_imgs, train_masks, image_size=IMG_SIZE, batch_size=BATCH_SIZE, is_train=True)
    val_dataset = create_tf_dataset(test_imgs, test_masks, image_size=IMG_SIZE, batch_size=BATCH_SIZE, is_train=False) if test_imgs else None
    total_batches = len(train_imgs) // BATCH_SIZE
    best_val_loss = float("inf")
    
    for epoch in range(1, TRAIN_EPOCHS + 1):
        epoch_loss = 0.0
        batches = 0
        t0 = time.time()
        
        for step, (x_b, y_m, y_g) in enumerate(train_dataset):
            loss = train_step(x_b, y_m, y_g)
            loss_val = float(loss)
            epoch_loss += loss_val
            batches += 1
            
            # Live batch-level MLflow metric logging every 10 batches
            if batches % 10 == 0:
                global_step = (epoch - 1) * total_batches + batches
                mlflow_tracker.log_metrics({
                    "batch_train_loss": loss_val,
                    "running_avg_train_loss": epoch_loss / batches
                }, step=global_step)
                
            if batches % 25 == 0 or batches == total_batches:
                print(f"Epoch [{epoch:02d}/{TRAIN_EPOCHS:02d}] Batch [{batches:03d}/{total_batches:03d}] | Loss: {loss_val:.4f} | Avg: {epoch_loss/batches:.4f}")
                
        avg_train_loss = epoch_loss / max(batches, 1)
        elapsed = time.time() - t0
        
        # Run validation pass
        val_loss_total = 0.0
        val_batches = 0
        if val_dataset:
            for vx_b, vy_m, vy_g in val_dataset:
                v_loss, _ = val_step(vx_b, vy_m, vy_g)
                val_loss_total += float(v_loss)
                val_batches += 1
        avg_val_loss = val_loss_total / max(val_batches, 1) if val_batches > 0 else avg_train_loss
        
        print(f"--> Epoch {epoch} Complete | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Time: {elapsed:.1f}s")
        
        # Log epoch-level metrics to MLflow
        mlflow_tracker.log_metrics({
            "epoch_train_loss": avg_train_loss,
            "epoch_val_loss": avg_val_loss,
            "epoch_time_seconds": elapsed,
            "learning_rate": float(lr_schedule(epoch * total_batches))
        }, step=epoch)
        
        # Save latest and best epoch checkpoints
        latest_ckpt = CHECKPOINT_DIR / f"dgnet_{MODEL_VERSION}_latest.weights.h5"
        model.save_weights(str(latest_ckpt))
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_ckpt = CHECKPOINT_DIR / f"dgnet_{MODEL_VERSION}_best.weights.h5"
            model.save_weights(str(best_ckpt))
            mlflow_tracker.log_artifact(str(best_ckpt), artifact_path="checkpoints")
            print(f"  ★ Best model saved: {best_ckpt.name} (Val Loss: {best_val_loss:.4f})")
else:
    print("ℹ Training loop skipped (set RUN_TRAINING = True to execute training).")


## 10. Benchmark Evaluation on Testing Split & MLflow Metrics Logging
Evaluates checkpoint performance on the test split, calculates official COD benchmark metrics, and logs them directly to the active MLflow run:
- `eval_pixel_accuracy`, `eval_mae`, `eval_iou`, `eval_dice`, `eval_f_measure` ($F_{0.3}$), `eval_precision`, `eval_recall`, `eval_specificity`, `eval_fps`.


In [ ]:
# Load best available weights for evaluation
candidate_checkpoints = sorted(CHECKPOINT_DIR.glob("*.weights.h5"), reverse=True)
if candidate_checkpoints:
    best_ckpt = candidate_checkpoints[0]
    _ = model(dummy_input, training=False)
    model.load_weights(str(best_ckpt))
    ckpt_stem = best_ckpt.name.replace(".weights.h5", "").replace(".h5", "")
    print(f"✓ Loaded weights checkpoint for evaluation: {best_ckpt.name}")
else:
    ckpt_stem = f"dgnet_{MODEL_VERSION}"
    print("ℹ No saved weights found; evaluating with initialized model weights.")

if test_imgs:
    test_dataset = create_tf_dataset(test_imgs, test_masks, image_size=IMG_SIZE, batch_size=4, is_train=False)
    metrics_acc = CODMetricsAccumulator(threshold=0.5, beta_sq=0.3)
    t_start = time.perf_counter()
    
    for x_b, y_m, _ in test_dataset:
        pred_out = model(x_b, training=False)
        # Handle both tensor and dictionary model outputs
        if isinstance(pred_out, dict):
            pred_mask = tf.nn.sigmoid(pred_out["mask_pred"])
        elif isinstance(pred_out, (list, tuple)):
            pred_mask = tf.nn.sigmoid(pred_out[0])
        else:
            pred_mask = tf.nn.sigmoid(pred_out) if pred_out.shape[-1] == 1 else pred_out
        metrics_acc.update(y_m, pred_mask)
        
    total_eval_time = time.perf_counter() - t_start
    eval_results = metrics_acc.result()
    fps = eval_results["sample_count"] / total_eval_time if total_eval_time > 0 else 0.0
    
    # Log evaluation metrics to MLflow
    mlflow_tracker.log_metrics({
        "eval_pixel_accuracy": float(eval_results["pixel_accuracy"]),
        "eval_mae": float(eval_results["mae"]),
        "eval_iou": float(eval_results["iou"]),
        "eval_dice": float(eval_results["dice"]),
        "eval_f_measure": float(eval_results["f_measure"]),
        "eval_precision": float(eval_results["precision"]),
        "eval_recall": float(eval_results["recall"]),
        "eval_specificity": float(eval_results["specificity"]),
        "eval_fps": float(fps)
    })
    
    print("\n" + "=" * 65)
    print("           DGNet Benchmark Evaluation Results")
    print("=" * 65)
    print(f" Model Evaluated:         {ckpt_stem}")
    print(f" Samples Evaluated:       {eval_results['sample_count']}")
    print("-" * 65)
    print(f" Pixel Accuracy:          {eval_results['pixel_accuracy']*100:.2f}%")
    print(f" Mean Absolute Error:     {eval_results['mae']:.4f}")
    print(f" IoU (Jaccard Index):     {eval_results['iou']:.4f}")
    print(f" Dice Coefficient (F1):   {eval_results['dice']:.4f}")
    print(f" F-measure (Fβ=0.3):     {eval_results['f_measure']:.4f}")
    print("-" * 65)
    print(f" Precision:               {eval_results['precision']:.4f}")
    print(f" Recall:                  {eval_results['recall']:.4f}")
    print(f" Specificity:             {eval_results['specificity']:.4f}")
    print(f" Realtime Speed:          {fps:.1f} FPS")
    print("=" * 65)
    print("✓ Logged all evaluation metrics to active MLflow run.")


## 11. Clean Black-and-White Visual Comparison & MLflow Artifact Logging
Generates pure **black-and-white binary masks** (no color heatmaps or overlays):
- `{stem}_gt.png`: Ground Truth (0 = background, 255 = target)
- `{stem}_{model}_pred.png`: Model prediction (0 and 255)
- `{stem}_{model}_comparison.png`: Side-by-side `[ GROUND TRUTH | MODEL: {name} ]`
The comparison images are automatically registered as **MLflow run artifacts** under `prediction_comparisons/`.


In [ ]:
def generate_and_save_clean_visual_comparison(sample_idx=0, save_dir=PRED_DIR, model_tag=ckpt_stem):
    if sample_idx >= len(test_imgs): return
    img_path = Path(test_imgs[sample_idx])
    gt_path = Path(test_masks[sample_idx])
    stem = img_path.stem
    
    img_raw = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, IMG_SIZE)
    img_float = (img_resized.astype(np.float32) / 255.0 - IMAGENET_MEAN.numpy()) / IMAGENET_STD.numpy()
    
    gt_raw = cv2.imread(str(gt_path), cv2.IMREAD_GRAYSCALE)
    gt_bin = (cv2.resize(gt_raw, IMG_SIZE, interpolation=cv2.INTER_NEAREST) >= 128).astype(np.uint8) * 255
    
    inp_tensor = tf.expand_dims(tf.convert_to_tensor(img_float, dtype=tf.float32), 0)
    pred_out = model(inp_tensor, training=False)
    if isinstance(pred_out, dict):
        pred_prob = tf.nn.sigmoid(pred_out["mask_pred"]).numpy()[0, :, :, 0]
    elif isinstance(pred_out, (list, tuple)):
        pred_prob = tf.nn.sigmoid(pred_out[0]).numpy()[0, :, :, 0]
    else:
        pred_prob = tf.nn.sigmoid(pred_out).numpy()[0, :, :, 0]
    pred_bin = (pred_prob >= 0.5).astype(np.uint8) * 255
    
    h, w = IMG_SIZE
    sep = np.full((h, 4), 128, dtype=np.uint8)
    body = np.hstack([gt_bin, sep, pred_bin])
    header = np.zeros((36, body.shape[1]), dtype=np.uint8)
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(header, "GROUND TRUTH", (15, 24), font, 0.5, 255, 1, cv2.LINE_AA)
    cv2.putText(header, f"MODEL: {model_tag}", (w + 19, 24), font, 0.5, 255, 1, cv2.LINE_AA)
    h_div = np.full((2, body.shape[1]), 128, dtype=np.uint8)
    comparison_img = np.vstack([header, h_div, body])
    
    gt_file = save_dir / f"{stem}_gt.png"
    pred_file = save_dir / f"{stem}_{model_tag}_pred.png"
    comp_file = save_dir / f"{stem}_{model_tag}_comparison.png"
    
    cv2.imwrite(str(gt_file), gt_bin)
    cv2.imwrite(str(pred_file), pred_bin)
    cv2.imwrite(str(comp_file), comparison_img)
    
    mlflow_tracker.log_artifact(str(comp_file), artifact_path="prediction_comparisons")
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(gt_bin, cmap="gray", vmin=0, vmax=255)
    axes[0].set_title("Ground Truth Mask", fontsize=12)
    axes[0].axis("off")
    axes[1].imshow(pred_bin, cmap="gray", vmin=0, vmax=255)
    axes[1].set_title(f"Prediction: {model_tag}", fontsize=12)
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
    
    print(f"✓ Saved clean GT:         {gt_file.name}")
    print(f"✓ Saved model prediction: {pred_file.name}")
    print(f"✓ Saved B&W comparison:   {comp_file.name} (Logged to MLflow)")

if test_imgs:
    for idx in range(min(3, len(test_imgs))):
        generate_and_save_clean_visual_comparison(sample_idx=idx)


## 12. Edge Quantization (TFLite INT8) & MLflow Run Completion
Converts DGNet to TensorFlow Lite, saves strictly to `models/quantized/dgnet_s_int8.tflite`, logs the `.tflite` model artifact to MLflow, and marks the MLflow run as `FINISHED`.


In [ ]:
TFLITE_EXPORT_PATH = PROJECT_ROOT / "models/quantized/dgnet_s_int8.tflite"
TFLITE_EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Exporting DGNet to TFLite INT8: {TFLITE_EXPORT_PATH.resolve()}...")

concrete_func = tf.function(lambda x: model(x, training=False)).get_concrete_function(
    tf.TensorSpec([1, IMG_SIZE[0], IMG_SIZE[1], 3], tf.float32)
)

converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model_bytes = converter.convert()
with open(TFLITE_EXPORT_PATH, "wb") as f:
    f.write(tflite_model_bytes)

size_mb = TFLITE_EXPORT_PATH.stat().st_size / (1024 * 1024)
print(f"✓ SUCCESS: TFLite model exported successfully ({size_mb:.2f} MB).")

# Log quantized model artifact to MLflow
mlflow_tracker.log_artifact(str(TFLITE_EXPORT_PATH), artifact_path="quantized_models")
mlflow_tracker.end_run(status="FINISHED")
print("✓ MLflow Run marked as FINISHED.")


## 13. Synchronize Checkpoints & Logs to S3 / Cloud Storage (Matching `scripts/hf_sync.py`)
Optional cell to push the newly trained checkpoints and MLflow logs back to the remote S3 / Hugging Face Storage Bucket.


In [ ]:
def upload_checkpoints_and_logs_to_s3(repo_id=HF_REPO_ID, hf_token=HF_TOKEN):
    """Uploads local checkpoints and MLflow logs to S3 / HF remote bucket matching scripts/hf_sync.py."""
    if not hf_token:
        print("Notice: HF_TOKEN / AWS secret not provided. Skipping remote upload.")
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=hf_token.strip())
        api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
        
        # Upload checkpoints
        for ckpt in CHECKPOINT_DIR.glob("*.weights.h5"):
            print(f"  • Uploading checkpoint {ckpt.name}...")
            api.upload_file(path_or_fileobj=str(ckpt), path_in_repo=f"models/raw_checkpoints/{ckpt.name}", repo_id=repo_id, repo_type="dataset")
            
        if TFLITE_EXPORT_PATH.exists():
            print(f"  • Uploading quantized model {TFLITE_EXPORT_PATH.name}...")
            api.upload_file(path_or_fileobj=str(TFLITE_EXPORT_PATH), path_in_repo=f"models/quantized/{TFLITE_EXPORT_PATH.name}", repo_id=repo_id, repo_type="dataset")
            
        # Upload MLflow DB if exists
        db_file = METRICS_DIR / "mlflow_db" / "mlflow.db"
        if db_file.exists():
            print(f"  • Uploading MLflow SQLite database...")
            api.upload_file(path_or_fileobj=str(db_file), path_in_repo="metrics/mlflow_db/mlflow.db", repo_id=repo_id, repo_type="dataset")
            
        print("✓ Remote S3 / HF Storage synchronization complete!")
    except Exception as e:
        print(f"Notice (S3 Push): {e}")

# Set PUSH_TO_S3 = True to upload artifacts after remote training
PUSH_TO_S3 = False
if PUSH_TO_S3:
    upload_checkpoints_and_logs_to_s3()


## 14. MLflow Tracking Database & UI Inspection
Queries the MLflow database (`metrics/mlflow_db/mlflow.db`) directly using `mlflow.search_runs()` and displays logged experiments history.


In [ ]:
mlflow.set_tracking_uri(mlflow_tracker.tracking_uri)
runs_df = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])

print(f"Total Runs Logged in '{EXPERIMENT_NAME}': {len(runs_df)}")
if not runs_df.empty:
    cols_to_show = [
        c for c in [
            "run_id", "tags.mlflow.runName", "status", "metrics.eval_iou",
            "metrics.eval_dice", "metrics.eval_pixel_accuracy", "metrics.eval_fps", "start_time"
        ] if c in runs_df.columns
    ]
    print("\nRecent MLflow Experiment Runs:")
    try:
        import pandas as pd
        pd.set_option('display.max_columns', 15)
        pd.set_option('display.width', 1000)
        print(runs_df[cols_to_show].head(5))
    except Exception:
        for idx, r in runs_df.head(5).iterrows():
            print(f"  • Run: {r.get('tags.mlflow.runName', r['run_id'])} | Status: {r['status']}")

print("\n" + "=" * 70)
print("                 To Launch the MLflow Web UI Locally / Remotely:")
print("=" * 70)
print(f"Run the following command in Terminal / PowerShell:")
print(f"  mlflow ui --backend-store-uri {mlflow_tracker.tracking_uri} --port 5000")
print("Then open your browser to: http://localhost:5000")
print("=" * 70)
